# 03 — XGBoost: Financial-Only Model

**Why XGBoost?** Gradient boosting on decision trees is the de-facto state of the art for tabular credit scoring since ~2015 (winning virtually every Kaggle credit competition since). It captures:

- **Non-linear patterns** logistic regression misses (e.g. risk jumps at specific thresholds of `past_due_90`)
- **Feature interactions** without explicit engineering (e.g. high utilization + young age compounding risk)
- **Missing values natively** via separate split directions (no imputation needed)

**Design decisions**:
- **Early stopping on val**: prevents overfitting automatically by stopping when val AUC stops improving.
- **`scale_pos_weight = neg/pos ≈ 14`**: compensates imbalance; XGBoost's equivalent of `class_weight='balanced'`.
- **Modest depth (5)**: deeper trees overfit on small positive class (only ~7k positives).
- **Low learning rate (0.05) + many estimators**: trades compute for stability. Early stopping picks the right stopping point.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

from src.data.split import load_splits
from src.evaluation.metrics import comparison_table, evaluate, plot_evaluation

plt.rcParams['figure.dpi'] = 100

splits = load_splits(Path.cwd().parent / 'data' / 'processed')
train, val, test = splits['train'], splits['val'], splits['test']
FEATURES = [c for c in train.columns if c != 'target']

X_train = train[FEATURES].astype({c: 'float64' for c in FEATURES})
X_val   = val[FEATURES].astype({c: 'float64' for c in FEATURES})
X_test  = test[FEATURES].astype({c: 'float64' for c in FEATURES})
y_train, y_val, y_test = train['target'].values, val['target'].values, test['target'].values

n_neg, n_pos = int((y_train == 0).sum()), int((y_train == 1).sum())
scale_pos_weight = n_neg / n_pos
print(f'Train: {n_neg:,} negative / {n_pos:,} positive -> scale_pos_weight = {scale_pos_weight:.2f}')

## Train with early stopping

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f'Best iteration: {model.best_iteration}')
print(f'Best val AUC: {model.best_score:.4f}')

In [ ]:
proba_train = model.predict_proba(X_train)[:, 1]
proba_val   = model.predict_proba(X_val)[:, 1]

r_train = evaluate(y_train, proba_train, 'XGB-fin', 'train')
r_val   = evaluate(y_val,   proba_val,   'XGB-fin', 'val')
comparison_table([r_train, r_val])

In [ ]:
fig = plot_evaluation(y_val, proba_val, 'XGBoost (financial only) — Validation')
plt.show()

## Calibration — can we trust the raw scores as probabilities?

XGBoost optimizes log-loss but isn't guaranteed to be calibrated out of the box, especially under heavy class reweighting (`scale_pos_weight`). Let's check.

A well-calibrated score of 0.10 means 10% of such cases actually default. For credit operations (pricing, provisioning, limits), this matters more than rank ordering alone.

In [ ]:
# Calibrate using isotonic regression on the val set (cv='prefit')
calibrated = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
calibrated.fit(X_val, y_val)

proba_val_cal = calibrated.predict_proba(X_val)[:, 1]
r_val_cal = evaluate(y_val, proba_val_cal, 'XGB-fin-cal', 'val')

comparison_table([r_val, r_val_cal])

Isotonic preserves AUC/KS (they're rank-based), but the Brier score should drop significantly — that's the calibration gain. In production, the calibrated model is what would be deployed for probability-based decisions.

In [ ]:
fig = plot_evaluation(y_val, proba_val_cal, 'XGBoost calibrated — Validation')
plt.show()

## Comparison with baseline

The hard question: does XGBoost beat the logistic regression baseline with enough margin to justify the added complexity? In credit, the answer isn't always yes — if the lift is small, interpretability wins.

In [ ]:
# Load baseline metrics from the metrics log
metrics_log_path = Path.cwd().parent / 'models' / 'metrics_log.json'
log = json.loads(metrics_log_path.read_text())

rows = [
    log['baseline_logreg']['val'],
    r_val.to_row(),
    r_val_cal.to_row(),
]
pd.DataFrame(rows)

**Expected takeaway**: XGBoost gives +4-5 points AUC and +7-8 points KS over the baseline. That's meaningful in credit — a KS jump from 50 to 58 typically translates to several percentage points of default-rate reduction at a fixed approval rate.

In [ ]:
# Feature importance (gain-based)
importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance['feature'], importance['importance'], color='#4C72B0')
ax.set_xlabel('Gain importance')
ax.set_title('XGBoost — feature importance (gain)')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

Gain-based importance is informative but **not the same as SHAP**. SHAP gives direction (positive vs negative effect per instance) and is additive across features. We do full SHAP analysis in `05_shap_analysis.ipynb`.

In [ ]:
# Persist model and metrics
models_dir = Path.cwd().parent / 'models'
joblib.dump(model, models_dir / 'xgboost_financial.pkl')
joblib.dump(calibrated, models_dir / 'xgboost_financial_calibrated.pkl')
print(f'Saved: xgboost_financial.pkl + xgboost_financial_calibrated.pkl')

log['xgboost_financial'] = {
    'train': r_train.to_row(),
    'val': r_val.to_row(),
    'val_calibrated': r_val_cal.to_row(),
    'best_iteration': int(model.best_iteration),
    'params': {
        'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 5,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'scale_pos_weight': round(scale_pos_weight, 3),
    },
}
metrics_log_path.write_text(json.dumps(log, indent=2))
print('metrics_log.json updated.')